## Notebook 8: Dynamic CBA

<blockquote style="border-left:4px solid #ccc; padding-left:1em;">
Here we take the city-specific outputs from previous notebooks (hazard, exposure, GVI uplift, AC coverage, CLIMADA runs) and build time profiles of adaptation costs for trees and AC (CAPEX + O&M), time profiles of benefits (avoided heat-related deaths), aggregate everything over a 25-year horizon with a 3% discount rate and derive equivalent annual costs (EACs) and cost-per-avoided-death metrics
</blockquote>

In [1]:
import os
os.environ["CITY"] = "rome"   # pick the city here

In [2]:
# Generic bootstrap 
from pathlib import Path
import os, sys

def _find_root():
    start = Path.cwd()
    for cand in [start, *start.parents]:
        if (cand/"cityheat").is_dir() and (cand/"configs").is_dir():
            return cand
    raise RuntimeError("Repo root not found.")
ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cityheat.nbsetup import bootstrap
from cityheat.paths import make_P, ensure_out

# Choose city here
SLUG = globals().get("SLUG", os.environ.get("CITY", "rome")).lower()

C    = bootstrap(SLUG)      # reads configs/<slug>.yml and syncs that city only if wanted
CFG  = C["CFG"]; CITY = C["CITY"]
BASE = C["BASE"]; OUT = C["OUT"]; INT = C["INT"]

P    = make_P(BASE)         # read-only path helper
OUTP = ensure_out(OUT)      # write-safe path helper
print(f"→ City: {CITY}  |  BASE={BASE}  OUT={OUT}  INT={INT}")

→ City: rome  |  BASE=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome  OUT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome  INT=/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/interim


In [3]:
# city config + file paths from YAML 
cfg = C.get("cfg", {})                      # full YAML for the selected city
SLUG = cfg.get("slug", SLUG).lower()        
CITY = cfg.get("city_name", CITY)

paths    = cfg.get("files", {})             # {gvi_csv, lcz_candidates, cooling_coeffs_csv, ...}
osm_cfg  = cfg.get("osm", {})               # OSM settings used later in NB5
trees_cfg = cfg.get("trees", {})            # TARGET/CAP for NB5
urbclim   = cfg.get("urbclim", {})          # UrbClim folder/settings for NB5

lcz_candidates = [P(p) for p in paths.get("lcz_candidates", [])]
gvi_path = P(paths.get("gvi_csv", ""))

# FUA geopackage written in NB2 
fua_gpkg = Path(paths.get("fua_gpkg", f"{OUT}/{SLUG}_fua.gpkg"))

cool_csv = P(paths.get("cooling_coeffs_csv", ""))

# checks
print("SLUG/CITY:", SLUG, CITY)
print("GVI CSV:  ", gvi_path)
print("LCZ cand: ", [str(p) for p in lcz_candidates])
print("FUA GPKG: ", fua_gpkg)
print("Cooling CSV:", cool_csv)

SLUG/CITY: rome Rome
GVI CSV:   /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/gviRome/gvi_Rome.csv
LCZ cand:  ['/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_filter_v3.tif', '/Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/LCZ/lcz_v3.tif']
FUA GPKG:  /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/outputs/rome/rome_fua.gpkg
Cooling CSV: /Users/armandeaboudrar-meda/Desktop/CMCC/URBADAPT/urban-heat/data/rome/CoolingEff/outer_2_wbgt_max.csv


**Loading from before**

We now load all the inputs that previous notebooks produced:
- trees_tbl: municipio-level tree policy and resulting ΔGVI
- veg_diag: diagnostic JSON with citywide pop-weighted ΔGVI (1–100 index)
- ac_coverage_maps_{SLUG}.npz: baseline vs policy AC coverage on the grid
- pop_on_ref_{SLUG}.npz: population on the common reference grid
- muni_cov_{SLUG}.csv: municipio-level AC coverage (baseline vs policy)
- {SLUG}_muni_ac_consumption_summary.csv: kWh per AC user by municipio, year
- ac_eff_buckets_{SLUG}.json: relative AC efficacy by age class (for benefits)

In [4]:
# loading everything needed for the CBA 
from pathlib import Path
import json
import numpy as np
import pandas as pd

OUT = Path(OUT)
INT = Path(INT)

TAB_DIR = OUT / "tables"

# Vegetation policy / ΔGVI 
trees_tbl = pd.read_csv(TAB_DIR / f"{SLUG}_trees_tbl.csv")

# diagnostics JSON with citywide ΔGVI (pop-weighted, points on 1–100 scale)
veg_diag_path = OUT / f"{SLUG}_veg_diagnostics.json"
veg_diag = json.loads(veg_diag_path.read_text())
citywide_dGVI_points_popw = veg_diag["citywide_dGVI_points_popw"]
print("Citywide pop-weighted ΔGVI (points, 1–100 scale):", citywide_dGVI_points_popw)

# AC coverage / municipal pop / energy use 
# coverage maps
ac_cov_npz = np.load(INT / f"ac_coverage_maps_{SLUG}.npz")
coverage_base   = ac_cov_npz["coverage_base"]
coverage_policy = ac_cov_npz["coverage_policy"]
CITY_MASK       = ac_cov_npz["CITY_MASK"].astype(bool)
HGT             = int(ac_cov_npz["HGT"])
WDT             = int(ac_cov_npz["WDT"])

# population on ref grid
pop_npz = np.load(INT / f"pop_on_ref_{SLUG}.npz")
pop_on_ref = pop_npz["pop"]

# municipio coverage table (pop_muni, ac_base_muni, ac_policy_muni)
muni_cov = pd.read_csv(OUT / f"muni_cov_{SLUG}.csv")

# AC consumption per municipality (kWh per AC user)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

# AC efficacy by age (for benefits, not directly cost-side) 
eff_buckets_path = INT / f"ac_eff_buckets_{SLUG}.json"
EFF_BUCKETS = json.loads(eff_buckets_path.read_text())
EFF_BUCKETS

Citywide pop-weighted ΔGVI (points, 1–100 scale): 2.3197


{'<15': 0.2, '15-64': 0.3, '65+': 0.4}

**Discount helpers**

We assume:
- Time horizon T = 25 years
- Constant real discount rate r = 3%
- All flows occur at the end of each year (years 1..T).

- Helper functions:
    - pv_level_flow: present value (PV) of a constant annual flow over T years.
    - pv_replacements: PV of buying an item at t=0 and then replacing it every 'life'
      years within the horizon.
    - annuity_factor: present value of "1 euro per year" over T years. We later use this
      to convert any PV into a constant equivalent annual cost (EAC).
    - pv_capex_with_ramp: PV of AC capex when new users are added year by year(ramp-up),
      with replacements every `life` years.

- Intuition:
    - First we build the actual time path of costs (trees and AC).
    - Then we discount and sum to get a PV.
    - Finally, we divide the PV by the annuity factor to get a flat annual amount that is
      financially equivalent to the time-varying cashflow.

In [5]:
import numpy as np

R = 0.03   # discount rate
T = 25     # time horizon

def pv_level_flow(annual, r=R, T=T):
    """PV of a constant annual amount paid in years 1..T."""
    yrs = np.arange(1, T+1, dtype=float)
    return float(np.sum(annual * (1 + r) ** (-yrs)))

def pv_replacements(n_items, capex_per_item, r=R, T=T, life=20):
    """
    PV of buying 'n_items' at t=0 and replacing every 'life' years within horizon T.
    """
    pv = 0.0
    t = 0
    while t <= T:
        pv += n_items * capex_per_item / ((1+r)**t if t > 0 else 1.0)
        t += life
    return float(pv)

def annuity_factor(r=R, T=T):
    return (1 - (1 + r) ** (-T)) / r

AF = annuity_factor(R, T)
AF

17.413147691278027

In [6]:
def pv_capex_with_ramp(new_users_t, capex_per_user, life, r=R):
    """
    PV of AC capex when policy coverage ramps up over time.

    new_users_t: 1D array length T, number of new policy users in each year (vs baseline).
    capex_per_user: installation cost per AC user.
    life: years between replacements.
    r: discount rate.

    For each cohort of new users in year t, we:
      - pay capex once at installation (year t+1 in our convention)
      - then pay the same capex again every `life` years (replacement)
      - discount each of these payments back to year 0 and sum them up.
    """
    T = len(new_users_t)
    pv = 0.0
    for t in range(T):   # t = 0..T-1 corresponds to years 1..T
        cohort = float(new_users_t[t])
        if cohort <= 0:
            continue
        pay_year = t
        # installation in year t+1, then every 'life' years
        while pay_year < T:
            pv += cohort * capex_per_user / ((1 + r) ** (pay_year + 1))
            pay_year += life
    return float(pv)

**Trees: parametrisation of costs**

Some explanation on following code:
- Total index : how many GVI index points we need in total (once and for all) to reach policy target
- Investment rule: each new index point costs 10M once (investment), not every year. Spreading creation of those index points linearly over 25y : each year we create the same slice of change in GVI and pay 10M*that slice in that year. That's the change in GVI = sum(change in GVI/25) and 10M * change in GVI/25 per year.
- CAPEX PV: npv_capex_linear: takes ramp of yearly investments (same amount each year, over 25 years) and discounts them as flows in years 1...25. PV_trees_capex: NPV of the linear capex schedule
- IR vs EAC: TREES_CAPEX_T0: total undiscounted investment requirement
- Report EAC_capex_annuity=PV_trees_capex/AF : equivalent annual cost of our explicit linear ramp and EAC_capex_paper=TREES_CAPEX_T0/(1+R)^T*T) : as if all investments happens by T and we just annualise that discounted IR.
- O&M: npv_om_cohorts: treats O&M as yearly flows in years 1...25 each year we add a new cohort, and the number of active cohorts in year t is min(t, lifetime). We pay O&M per index point per year for all active cohorts and discount those flows. Matches description of O&M series that starts when trees are planted, accumulates cohorts, and is discounted. 

**Timing conventions for tree costs.**  
We consider a 25-year horizon and treat all cash flows as occurring at the end of each year (years 1–25). The total increase in the GVI index (ΔGVI, in 1–100 index points) implied by the tree policy is first converted into a **total investment requirement** using the rule that raising GVI by 1 index point costs 10 million euro once and for all. We then assume this investment is implemented along a **linear ramp**: the same fraction of ΔGVI is created in each year, and the corresponding investment is spread evenly over the 25 years. The functions `npv_capex_linear` and `npv_om_cohorts` compute the discounted present value of, respectively, this phased investment schedule and the recurrent O&M costs associated with overlapping planting cohorts. From these present values we derive equivalent annual costs (EACs) by dividing by the standard annuity factor. For comparison with the article (working paper), we also report a “paper-style” EAC where the total undiscounted investment requirement is pushed to the end of the horizon, discounted once, and then divided by the number of years.

**Trees: dynamic CAPEX + O&M costs for a given GVI uplift**

- Policy:
    - We simulate a tree-planting policy that increases the Green View Index (GVI)
    in low-GVI municipi up to a target level.
    - DELTA_INDEX is the total citywide increase in GVI, measured in 1–100 index points,
      implied by this policy (sum over municipi of dGVI * 100).
- Cost rule:
    - We use an empirical rule: increasing GVI by 1 index point costs €10 million
      once and for all (investment) (working paper)
    - TREES_CAPEX_T0 = 10 M€ * DELTA_INDEX is the undiscounted total investment
      requirement if we imagined doing all planting at once.

- Time pattern (ramp):
    - In reality we don't plant everything in one year. Instead we assume a
      linear rollout over T = 25 years:
      each year we create the same slice of ΔGVI (DELTA_INDEX / T)
      each slice pays CAPEX once at planting in that year
    - npv_capex_linear():
      builds this stream of annual investments (same € amount each year)
      discounts each year's CAPEX to year 0
      sums to a present value PV_trees_capex

- O&M (operation and maintenance):
    - Tree costs do not end at planting: each "cohort" of trees needs O&M every year.
    - We use REGREEN per-tree numbers (CAPEX = 210 €, O&M = 27 €/yr) to derive an
      O&M cost per GVI index point per year (OM_PER_INDEX_PT_YR).
    - npv_om_cohorts():
      assumes we add the same GVI increment each year (cohorts)
      each cohort pays OM_PER_INDEX_PT_YR * increment every year after planting
      up to LIFETIME_YEARS (25 here)
      at year t, there are min(t, LIFETIME_YEARS) active cohorts
      discounts the resulting O&M stream to year 0 to get PV_trees_om

- Aggregation and annualisation:
    - PV_trees_total = PV_trees_capex + PV_trees_om is the total present value of the
      tree programme (investment + O&M).
    - We convert each PV into an equivalent annual cost by dividing by the annuity factor
      AF:
      EAC_capex_annuity = PV_trees_capex / AF
      EAC_om_annuity    = PV_trees_om    / AF
      EAC_total_annuity = PV_trees_total / AF

- Paper-style EAC (for comparability only):
    - EAC_capex_paper uses the shortcut from the original paper:
  take TREES_CAPEX_T0 (total undiscounted investment requirement),
  pretend it is paid all at year T,
  discount once by (1 + r)^T,
  ivide by T.
    - This does NOT reflect our explicit linear rollout => gives an "average discounted annual investment" from a single IR number.
    - The economically consistent metric for our dynamic ramp is the annuity-based
    - EAC derived from PV_trees_capex and PV_trees_om.

In [7]:
# Parameters from rule and regreen study 
CAPEX_PER_INDEX_PT = 10_000_000.0   # eur per 1 index point (1–100 scale)
CAPEX_PER_TREE     = 210.0          # eur per tree (REGREEN median)
OM_PER_TREE_YR     = 27.0           # eur per tree per year
LIFETIME_YEARS     = 25             # tree benefit/O&M lifetime 

# O&M per index point per year implied by tree-level numbers
OM_PER_INDEX_PT_YR = (OM_PER_TREE_YR / CAPEX_PER_TREE) * CAPEX_PER_INDEX_PT
print("O&M per index point per year (EUR):", round(OM_PER_INDEX_PT_YR, 0))

# Total change in GVI in index point (1–100 SCALE) from trees_tbl 
# trees_tbl['dGVI'] is in 0–1 (fraction of max index); sum over Municipi, then ×100 => points
DELTA_INDEX = float(trees_tbl["dGVI"].clip(lower=0).sum()) * 100.0  # sum of municipio dGVI (0–1) => index points (0–100)
print("Total ΔGVI index points (1–100 scale):", round(DELTA_INDEX, 2))

# For comparison: citywide pop-weighted ΔGVI (diagnostics)
print("Pop-weighted ΔGVI points (diagnostic):", citywide_dGVI_points_popw)

# CAPEX: linear ramp over T years 
def npv_capex_linear(delta_index_total, years=T, r=R,
                     capex_per_index=CAPEX_PER_INDEX_PT):
    """
    PV of a linear ramp: we add delta_index_total/years index points
    each year over 'years', and pay capex_per_index per point.
    All flows are assumed at the end of years 1..years.
    """
    inc = delta_index_total / years  # index points added per year
    pv = 0.0
    for t in range(1, years + 1):    # t = 1..years
        capex_t = capex_per_index * inc
        pv += capex_t / ((1 + r) ** t)
    return float(pv)

TREES_CAPEX_T0 = CAPEX_PER_INDEX_PT * DELTA_INDEX   # undiscounted total “once and for all” cost
PV_trees_capex = npv_capex_linear(DELTA_INDEX, years=T, r=R, capex_per_index=CAPEX_PER_INDEX_PT)

print(f"Trees — Total CAPEX requirement (undiscounted): €{TREES_CAPEX_T0:,.0f}")
print(f"Trees — NPV CAPEX (linear ramp):              €{PV_trees_capex:,.0f}")

# O&M: overlapping cohorts with constant per-index-point O&M 
def npv_om_cohorts(delta_index_total, years=T, r=R,
                   om_per_index_per_year=OM_PER_INDEX_PT_YR, lifetime=LIFETIME_YEARS):
    """
    O&M with overlapping cohorts: each year we add 'inc' index points
    and each cohort pays om_per_index_per_year * inc every year after planting,
    up to 'lifetime' years. All flows at the end of years 1..years.
    """
    inc = delta_index_total / years
    pv = 0.0
    for t in range(1, years + 1):    # t = 1..years
        active = min(t, lifetime)    # number of active cohorts in year t
        om_t = active * om_per_index_per_year * inc
        pv += om_t / ((1 + r) ** t)
    return float(pv)

PV_trees_om = npv_om_cohorts(DELTA_INDEX, years=T, r=R,
                             om_per_index_per_year=OM_PER_INDEX_PT_YR,
                             lifetime=LIFETIME_YEARS)

print(f"Trees — NPV O&M (cohorts):                   €{PV_trees_om:,.0f}")

# program totals 
PV_trees_total = PV_trees_capex + PV_trees_om
print(f"Trees — NPV total (CAPEX + O&M):             €{PV_trees_total:,.0f}")

# EACs: standard annuity vs paper-style formula 
AF = annuity_factor(R, T)

EAC_capex_annuity = PV_trees_capex / AF
EAC_om_annuity    = PV_trees_om    / AF
EAC_total_annuity = PV_trees_total / AF

# Article-like “investment requirement divided by (1+r)^T * T”
EAC_capex_paper = TREES_CAPEX_T0 / ((1 + R)**T * T)

print(f"Trees — EAC CAPEX (annuity):   €{EAC_capex_annuity:,.0f}/yr")
print(f"Trees — EAC O&M (annuity):     €{EAC_om_annuity:,.0f}/yr")
print(f"Trees — EAC total (annuity):   €{EAC_total_annuity:,.0f}/yr")
print(f"Trees — EAC CAPEX (paper-style): €{EAC_capex_paper:,.0f}/yr")

O&M per index point per year (EUR): 1285714.0
Total ΔGVI index points (1–100 scale): 30.23
Pop-weighted ΔGVI points (diagnostic): 2.3197
Trees — Total CAPEX requirement (undiscounted): €302,333,765
Trees — NPV CAPEX (linear ramp):              €210,583,300
Trees — NPV O&M (cohorts):                   €310,733,610
Trees — NPV total (CAPEX + O&M):             €521,316,910
Trees — EAC CAPEX (annuity):   €12,093,351/yr
Trees — EAC O&M (annuity):     €17,844,770/yr
Trees — EAC total (annuity):   €29,938,120/yr
Trees — EAC CAPEX (paper-style): €5,775,852/yr


- DELTA_INDEX: citywide change in GVI on 1-100 scale
- inc = DELTA_INDEX/years is change in GVI / 25 each year => linear path
- capex_t = 10Meur * inc is the 10Meur*change in GVI/25 rule
- Discounting stream year by year to get PV_trees_capex
- TREES_CAPEX_T0: paper way of doing it: what if we did everything upfront

- each year we plant inc index points (new cohort)
- each cohort costs om_per_index_per_year * inc every year
- after t years, there are t+1 overlapping cohorts (until we cap at lifetime)
- om_t is exactly sum over all active cohorts
- then we discount om_t year by year
- here, we choose that all cohorts have same per-index-point O&M each year for lifetime years

BIG QUESTION HERE BECAUSE I NEVER UNDERSTAND:
About the annualisation:
- We have two different annualisation ideas in the code
- 1. Economically consistent one with our ramp (PV_trees_capex, AF, EAC_capex_annuity)
     Interpretation: npv_capex_linear gives the present value of the phased CAPEX stream
     Dividing by the annuity factor (that only depends on T and r), converts that pV into
     a constant equivalent annual cost over 25 years.
     Basically, given a specific investment path (here: linear ramp), we discount it,
     then convert to PV into a flat annual amount.
  2. EAC_capex_paper: this is not the same thing as annuity based on ramp. It's like the
     paper of Giacomo: take undiscounted total investment requirement IR, push it to year
     T, discount it once, divide by T. It's to get the "average discounted annual cost"
     but assumes everything at the end and does not reflect our explicit tamp path. The
     correct one should be the 1. but need to ask Giacomo more about this. 

More explanation about this...
- the annuity EAC: takes the actual NPV of the ramped CAPEX (and O&M) streams, divide by the standard annuity factor. "Constant yearly cost, over 255 years, that is financially equivalent to this time varying cashflow"
- Paper : take the total undiscounted investment requirement (as if all invested once), shift it to year T then divide by T. It doesn't reflect our ramp. It's a shortcut to get an average discounted annual cost from a single IR number. We use it for comparability.

From what I understand, what I do is what we discussed: we have a total increase in GVI needed (delta_index), we assume a linear path over 25 years and each year we add change in GVI/25 index points. We use the rule: 1 index points: 10M once, not every year. So each year we invest 10M*(deltaGVI/25). We discount that year by year to get an NPV of capex: npv_capex_linear. For O&M, cohort logic: each year, new cohort planted, each cohort pays the same O&M per index point each year, in year t we have mint(, lifetime) active cohorts, we discount the whole panel of O&M flows: npv_om_cohorts. We do the standard annuity step : taking NPV of CAPEX (or total CAPEX + O&M) and dividing by annuity factor sum from t=1 to T of (1+r)^t to get a constant yearly cost that is financially equivalent to the detailed schedule. 

- npv_capex_linear: discounted sum of future cashflows with a linear ramp
- npv_om_cohorts: O&M cohorts idea, discounted year by year
- EAC by dividing by annuity factor. Depends only on r and T because it’s the present value of paying “1€ every year” over T years. It doesn’t have to know about how cohorts build up; that’s already into the NPV. 

What is happening in the paper? IRc is a total undiscounted investment requirement (already aggregated over the 25 y horizon). Then, you push that entire amount to year T, discount it once by (1+r)^T and divide by T. It's: taking the total investment we would need in today's euros, pretending it's all paid in year T, discounting that once, then just spreading the discounted lump evenly over T years. 

In the paper, the formula gave an “average discounted yearly investment” directly from a total IR, without specifying a time pattern.

Here, we now do specify the time pattern explicitly (linear ramp of ΔGVI and overlapping O&M cohorts). So I compute NPV as the discounted sum of those yearly cashflows, and then convert that NPV into a constant equivalent annual cost using the standard annuity factor for r = 3% and T=25.

This annuity-based EAC is the one that’s fully consistent with the dynamic ramp and with the way we treat benefits (yearly avoided deaths). I still report the paper-style EAC (IR/(1+r)^T/T) alongside, to keep direct comparability with your article.

**Sensitivity paved streets**

TO DO IF IT'S RIGHT, WITH 5379.0 instead of 210.

**Cost AC**

**Air conditioning (AC): dynamic coverage and dynamic electricity use** 

- Policy:
    - We expand AC coverage in under-served, poorer CAPs up to a target share.
    - The result is a time series of incremental AC coverage by municipio and year
      (policy vs baseline), coming from previous notebooks.
      
- Two key dynamics:
- 1) Coverage ramp:
    - muni_cov_all gives AC coverage for 2030, 2040, 2050 by municipio under baseline and
     policy.
    - For each municipio, we compute the additional AC share (policy - baseline) and
      interpolate it linearly year-by-year over 2030–2054.
    - Multiplying by pop_muni gives the number of additional AC users per year.
        - added_users_t = total incremental AC users in each year (vs baseline).
        - new_users_t   = number of *new* users added each year (increments of
          added_users_t) => drives CAPEX cohorts.
- 2) Electricity use per user:
     - NB7 gives kWh per AC user at city and municipio level for 2030, 2040, 2050.
     - We linearly interpolate kWh per user by municipio for each year of the
       25-year horizon (2030–2054).
     - For each municipio and year we then compute: kWh = pop_muni * extra AC share * kWh
       per user and aggregate to the city.
     
Cost components:
- CAPEX:
    - AC_CAPEX_PER_USER is the unit cost per new AC user.
    - pv_capex_with_ramp(new_users_t, ...) computes the present value of capex for each
      cohort, including replacements every AC_LIFETIME_YEARS.
- Maintenance:
    * Each incremental AC user pays a fixed fraction of CAPEX (5%) per year.
    * maint_eur_t = added_users_t * maint_per_user_yr gives yearly maintenance.
    * We discount this stream to get PV_ac_maint.
- Electricity:
    * elec_eur_t are yearly electricity costs (kWh * tariff) with both dynamic coverage
      and dynamic kWh/user.
    * We discount them to get PV_ac_elec.
      
- Aggregation and annualisation:
    - PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec.
    - EAC_ac_total = PV_ac_total / AF gives the equivalent constant annual cost over
      25 years of the full AC expansion (including CAPEX, maintenance and electricity).

- Interpretation:
- This AC module is fully dynamic: both the number of users and their per-user electricity consumption change over time, and CAPEX follows cohorts + replacements.
- This keeps AC costs consistent with the way we treat benefits (annual profiles over 25 years, discounted at 3%).

In [9]:
# Dynamic AC horizon years (aligned with NB7 / CLIMADA outputs)
# We only need the start year and the list of years for the CBA horizon.

ac_city_series = pd.read_csv(INT / f"ac_per_user_city_{SLUG}.csv")
ac_city_series = ac_city_series.set_index("year").sort_index()

ELEC_START_YEAR = int(ac_city_series.index.min())   # should be 2030
ELEC_YEARS      = np.arange(ELEC_START_YEAR, ELEC_START_YEAR + T, dtype=int)

In [10]:
# AC COSTS: dynamic coverage + dynamic kWh/user 

# AC PARAMS 
AC_CAPEX_PER_USER   = 500.0   # € per AC unit
AC_MAINT_RATE       = 0.05    # fraction of CAPEX per year
AC_LIFETIME_YEARS   = 10      # replacement cycle
TARIFF_EUR_PER_KWH  = 0.25    # €/kWh

# per-user annual maintenance 
maint_per_user_yr = AC_MAINT_RATE * AC_CAPEX_PER_USER

# Horizon years (should match benefits horizon: 2030..2030+T-1)
YEARS = ELEC_YEARS.copy()
assert len(YEARS) == T

# Municipio level coverage over time 
# Table with coverage by year and municipality (cf notebook 5, we have it there)
try:
    muni_cov_all = pd.read_csv(OUT / f"{SLUG}_muni_cov_yearly.csv")
except FileNotFoundError:
    muni_cov_all = muni_cov.copy()

# we keep only rows for which coverage is defined
# and, we can, restrict to Municipi inside the city proper
if "muni_id" in muni_cov_all.columns:
    muni_cov_all = muni_cov_all.loc[muni_cov_all["muni_id"] > 0].copy()

# additional AC share per municipio and year (policy vs baseline)
muni_cov_all["dshare"] = (
    muni_cov_all["ac_policy_muni"] - muni_cov_all["ac_base_muni"]
).clip(lower=0.0)

# interpolating dshare to all years for each municipality
rows = []
for muni_id, g in muni_cov_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years   = g["year"].to_numpy(int)
    known_dshare  = g["dshare"].to_numpy(float)
    pop_muni      = float(g["pop_muni"].iloc[0])   # assume pop_muni constant over time

    dshare_t = np.interp(YEARS, known_years, known_dshare)
    # flat before first and after last known year
    dshare_t[YEARS <= known_years[0]]  = known_dshare[0]
    dshare_t[YEARS >= known_years[-1]] = known_dshare[-1]

    for year, ds in zip(YEARS, dshare_t):
        rows.append({
            "year": year,
            "muni_id": muni_id,
            "pop_muni": pop_muni,
            "dshare_t": ds,
        })

# coverage ramp 
cov_yearly = pd.DataFrame(rows)

# total policy AC users per year (vs baseline)
added_users_t = (
    cov_yearly
    .assign(users=lambda d: d["pop_muni"] * d["dshare_t"])
    .groupby("year")["users"]
    .sum()
    .reindex(YEARS)
    .to_numpy(float)
)

# new users in each year (for capex cohorts)
new_users_t = np.empty_like(added_users_t)
new_users_t[0]  = added_users_t[0]
new_users_t[1:] = np.maximum(added_users_t[1:] - added_users_t[:-1], 0.0)

added_users_final = float(added_users_t[-1])
print(f"AC — added users in final year ≈ {added_users_final:,.0f}")

# Electricity costs: dynamic coverage + dynamic kWh/user from NB7 

# muni-level kWh per AC user from NB7 (available for 2030, 2040, 2050)
muni_tbl_all = pd.read_csv(OUT / f"{SLUG}_muni_ac_consumption_summary.csv")

years_full = YEARS  # array([2030, ..., 2054])

rows_kwh = []
for muni_id, g in muni_tbl_all.groupby("muni_id"):
    g = g.sort_values("year")
    known_years = g["year"].to_numpy(int)
    vals        = g["kwh_per_user_muni"].to_numpy(float)

    # interpolate to the full horizon
    kwh_interp = np.interp(years_full, known_years, vals)
    # flat before first and after last known year
    kwh_interp[years_full <= known_years[0]]  = vals[0]
    kwh_interp[years_full >= known_years[-1]] = vals[-1]

    for y, v in zip(years_full, kwh_interp):
        rows_kwh.append({
            "year": y,
            "muni_id": muni_id,
            "kwh_per_user_muni": v,
        })

muni_kwh_full = pd.DataFrame(rows_kwh)

# attach interpolated kWh/user to coverage ramp
cov_yearly = cov_yearly.merge(
    muni_kwh_full,
    on=["year", "muni_id"],
    how="left"
).fillna({"kwh_per_user_muni": 0.0})

# kWh per year: pop * extra AC share * kWh per AC user
cov_yearly["kwh_t"] = (
    cov_yearly["pop_muni"]
    * cov_yearly["dshare_t"]
    * cov_yearly["kwh_per_user_muni"]
)

elec_eur_t = (
    cov_yearly.groupby("year")["kwh_t"].sum()
    .reindex(YEARS)
    .to_numpy(float)
) * TARIFF_EUR_PER_KWH

# discounted PV of elec, capex, maintenance 

yrs = np.arange(1, T+1, dtype=float)   # 1..25

PV_ac_elec = float(np.sum(elec_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV elec (dynamic coverage & kWh/user): €{PV_ac_elec:,.0f}")

# maintenance on all active policy users
maint_eur_t = added_users_t * maint_per_user_yr
PV_ac_maint = float(np.sum(maint_eur_t * (1 + R) ** (-yrs)))
print(f"AC — PV maint €{PV_ac_maint:,.0f}")

# capex: cohorts of new users, with replacements every AC_LIFETIME_YEARS
PV_ac_capex = pv_capex_with_ramp(
    new_users_t,
    capex_per_user=AC_CAPEX_PER_USER,
    life=AC_LIFETIME_YEARS,
    r=R,
)
print(f"AC — PV capex €{PV_ac_capex:,.0f}")

# Total PV and EAC 

PV_ac_total = PV_ac_capex + PV_ac_maint + PV_ac_elec

print(f"AC — PV capex €{PV_ac_capex:,.0f}")
print(f"AC — PV maint €{PV_ac_maint:,.0f}")
print(f"AC — PV elec  €{PV_ac_elec:,.0f}")
print(f"AC — PV total €{PV_ac_total:,.0f}")

EAC_ac_total = PV_ac_total / AF
print(f"AC — EAC total (annuity): €{EAC_ac_total:,.0f}/yr")

AC — added users in final year ≈ 189,806
AC — PV elec (dynamic coverage & kWh/user): €410,141,921
AC — PV maint €75,254,250
AC — PV capex €212,003,495
AC — PV capex €212,003,495
AC — PV maint €75,254,250
AC — PV elec  €410,141,921
AC — PV total €697,399,665
AC — EAC total (annuity): €40,050,178/yr


In [11]:
print("First 5 years of elec_eur_t:", elec_eur_t[:5])
print("Last 5 years of elec_eur_t:", elec_eur_t[-5:])

First 5 years of elec_eur_t: [28357714.12672251 26831520.20637242 25334345.28124008 23866189.35132546
 22427052.41662859]
Last 5 years of elec_eur_t: [32657861.12709964 32657861.12709964 32657861.12709964 32657861.12709964
 32657861.12709964]


**Benefits and summary**

**Benefits: avoided heat-related deaths (CLIMADA outputs)** 

- Inputs:
    - CLIMADA provides avoided heat-related deaths for 2030, 2040, 2050 under:
      * Trees vs current AC baseline    (avo_trees)
      * AC policy vs baseline           (avo_ac)
      * Trees + AC policy vs current AC (avo_both)

- These are already *incremental* impacts relative to the current-AC baseline (and vegetation)

- Time profile:
    - We interpolate each of these three series to a yearly time profile over the
      25-year horizon (ex: 2030–2054).

- Trees vs current AC:
    - Trees do not deliver their full cooling effect immediately
    - We apply a smooth linear ramp over TREE_RAMP_YEARS (here 12 years) to the
      tree effect:
      * tree_ramp goes from 0 in year 1 to 1 by year 12
      * trees_yr_ramped = trees_full * tree_ramp

- AC vs baseline:
    - We assume the AC policy is *in place* from 2030 onwards (no ramp in benefits),
      so ac_yr is just the interpolated AC avoided-death series.

- Trees + AC:
    - both_full is the CLIMADA result for "trees + AC" vs the current AC baseline.
    - We decompose it as:
      * AC-only effect: ac_full
      * extra tree effect conditional on AC: trees_cond_yr = both_full - ac_full
    - The conditional tree effect is ramped in the same way:
      * both_yr_ramped = ac_yr + trees_cond_yr * tree_ramp

- Discounted vs cumulative benefits:
    - pv_of_stream(...) computes the *discounted* present value of each avoided-death
      stream (PV_b_tree, PV_b_ac, PV_b_both), which is useful for comparing timing of
      benefits.
      - For cost-per-avoided-death metrics, we use instead the *undiscounted* cumulative number of avoided deaths over the horizon:
        * CUM_b_tree, CUM_b_ac, CUM_b_both and then overwrite PV_b_* variables with these
          cumulative counts for compatibility with the rest of the code.

- Incremental effect of trees given AC:
  - PV_b_tree_cond and CUM_b_tree_cond capture the *extra* benefit of trees if the AC policy is already in place (both - AC).
  - This allows us to report a marginal cost-per-avoided-death for trees "on top" of an AC expansion.

In [12]:
# Benefits and summary

HORIZON_YEARS   = T
DISCOUNT_RATE   = R
TREE_RAMP_YEARS = 12

# smooth ramp 0 to 1 over TREE_RAMP_YEARS
tree_ramp = np.minimum(np.arange(1, HORIZON_YEARS+1) / TREE_RAMP_YEARS, 1.0)

def pv_of_stream(cashflows, r=DISCOUNT_RATE):
    yrs = np.arange(1, len(cashflows) + 1, dtype=float)
    return float(np.sum(np.asarray(cashflows, float) * (1 + r) ** (-yrs)))

import numpy as np
import pandas as pd
from pathlib import Path

INT = Path(INT)

# Trees (avoided deaths per year, trees vs current-AC baseline)
avo_trees = pd.read_csv(
    INT / f"annual_heat_deaths_avoided_trees_curr_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

# AC (policy vs baseline, overall avoided deaths per year) - Series indexed by year
avo_ac = pd.read_csv(
    INT / f"annual_heat_deaths_climada_avoided_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

# Trees + AC policy (vs current AC baseline) 
avo_both = pd.read_csv(
    INT / f"annual_heat_deaths_avoided_treesplusAC_curr_AC_{SLUG}.csv",
    index_col="year"
)["overall"]

print("Trees – avoided deaths per year:")
print(avo_trees, "\n")

print("AC – avoided deaths per year:")
print(avo_ac, "\n")

print("Both (trees+AC vs current AC) – avoided deaths per year:")
print(avo_both)

# building yearly benefit paths by interpolation between 2030/2040/2050
START_YEAR = int(min(avo_trees.index))
YEARS = np.arange(START_YEAR, START_YEAR + HORIZON_YEARS, dtype=int)

def interpolate_to_horizon(s, years):
    s = s.sort_index()
    known = s.index.to_numpy(int)
    vals  = s.to_numpy(float)
    out = np.interp(years, known, vals)
    out[years <= known[0]]  = vals[0]
    out[years >= known[-1]] = vals[-1]
    return out

# interpolate each scenario
trees_full = interpolate_to_horizon(avo_trees, YEARS)
ac_full    = interpolate_to_horizon(avo_ac,    YEARS)
both_full  = interpolate_to_horizon(avo_both,  YEARS)

# ramp
tree_ramp = np.minimum(np.arange(1, HORIZON_YEARS+1) / TREE_RAMP_YEARS, 1.0)

# Trees-only: ramp the full trees effect vs current AC
trees_yr        = trees_full
trees_yr_ramped = trees_yr * tree_ramp

# AC-only: no ramp
ac_yr = ac_full

# BOTH: AC immediate + tree effect ramp, but now the tree effect is conditional on AC
trees_cond_yr   = both_full - ac_full   # extra effect of trees when AC policy is present
both_yr_ramped  = ac_yr + trees_cond_yr * tree_ramp

# PV of avoided deaths
PV_b_tree = pv_of_stream(trees_yr_ramped, DISCOUNT_RATE)
PV_b_ac   = pv_of_stream(ac_yr,           DISCOUNT_RATE)
PV_b_both = pv_of_stream(both_yr_ramped,  DISCOUNT_RATE)

print("PV benefits (avoided deaths) —",
      f"Trees: {PV_b_tree:,.2f} | AC: {PV_b_ac:,.2f} | Both: {PV_b_both:,.2f}")

# sensitivity: undiscounted cumulative avoided deaths (no time weighting) 

CUM_b_tree = float(np.sum(trees_yr_ramped))   # trees: ramped benefits
CUM_b_ac   = float(np.sum(ac_yr))            # AC: no ramp (benefits immediate)
CUM_b_both = float(np.sum(both_yr_ramped))   # both: AC + ramped tree extra

print("Cumulative avoided deaths over horizon (undiscounted) — "
      f"Trees: {CUM_b_tree:,.2f} | AC: {CUM_b_ac:,.2f} | Both: {CUM_b_both:,.2f}")

# incremental trees benefit given AC policy
PV_b_tree_cond   = PV_b_both - PV_b_ac      # PV avoided deaths, trees on top of AC
CUM_b_tree_cond  = CUM_b_both - CUM_b_ac    # cumulative (undiscounted), trees on top of AC

Trees – avoided deaths per year:
year
2030    8.789494
2040    8.467261
2050    8.996251
Name: overall, dtype: float64 

AC – avoided deaths per year:
year
2030    24.525914
2040    20.229590
2050    26.856095
Name: overall, dtype: float64 

Both (trees+AC vs current AC) – avoided deaths per year:
year
2030    34.385183
2040    32.385640
2050    35.311887
Name: overall, dtype: float64
PV benefits (avoided deaths) — Trees: 109.62 | AC: 408.02 | Both: 537.72
Cumulative avoided deaths over horizon (undiscounted) — Trees: 170.73 | AC: 592.32 | Both: 790.00


In [13]:
# using UNDISCOUNTED avoided deaths for all ratios/tables below 
# cumulative avoided deaths instead of discounted PV of deaths
PV_b_tree      = CUM_b_tree
PV_b_ac        = CUM_b_ac
PV_b_both      = CUM_b_both
PV_b_tree_cond = CUM_b_tree_cond

- Baseline mortality and percentage reductions
- 
We now read the baseline heat-attributable deaths with current AC (no new policy) from CLIMADA and:
    - compute total baseline deaths per year (summing over age classes)
    - align avoided deaths for trees, AC, and both to these baseline years
    - compute percentage reductions in baseline deaths:
      * trees_pct = % reduction with trees only
      * ac_pct    = % reduction with AC only
      * both_pct  = % reduction with both policies
      * trees_on_top_pct = extra % reduction from adding trees on top of AC

In [14]:
from pathlib import Path
import pandas as pd

# baseline heat deaths with current AC (RAW CLIMADA, no scaling)
base_ac_path = Path(INT) / f"annual_heat_deaths_curr_AC_base_{SLUG}.csv"

tmp = pd.read_csv(base_ac_path)
print("Columns in current-AC baseline CSV:", list(tmp.columns))

# inferring year index
if "year" in tmp.columns:
    baseline_by_age = tmp.set_index("year")
else:
    baseline_by_age = tmp.set_index(tmp.columns[0])
    baseline_by_age.index.name = "year"

display(baseline_by_age)

# total baseline deaths 
if set(["<15", "15-64", "65+"]).issubset(baseline_by_age.columns):
    baseline_total = baseline_by_age[["15-64", "65+", "<15"]].sum(axis=1)
else:
    baseline_total = baseline_by_age["overall"]

baseline_total.name = "baseline_total"

print("Baseline total heat-attributable deaths per year (current AC, RAW):")
display(baseline_total)

# align avoided deaths with baseline years
avo_trees_al = avo_trees.reindex(baseline_total.index)
avo_ac_al    = avo_ac.reindex(baseline_total.index)
avo_both_al  = avo_both.reindex(baseline_total.index)

benefit_pct = pd.DataFrame({
    "baseline_total": baseline_total,
    "avo_trees":      avo_trees_al,
    "avo_ac":         avo_ac_al,
    "avo_both":       avo_both_al,
})

benefit_pct["trees_pct"] = 100 * benefit_pct["avo_trees"] / benefit_pct["baseline_total"]
benefit_pct["ac_pct"]    = 100 * benefit_pct["avo_ac"]    / benefit_pct["baseline_total"]
benefit_pct["both_pct"]  = 100 * benefit_pct["avo_both"]  / benefit_pct["baseline_total"]
benefit_pct["trees_on_top_pct"] = benefit_pct["both_pct"] - benefit_pct["ac_pct"]

benefit_pct.round(2)

Columns in current-AC baseline CSV: ['year', '<15', '15-64', '65+', 'overall']


,<15,15-64,65+,overall
year,,,,
2030,3.382917,77.091174,590.414426,670.888517
2040,3.276677,73.214870,551.934678,628.426226
2050,3.452808,78.970665,607.082165,689.505638


Baseline total heat-attributable deaths per year (current AC, RAW):


year
2030    670.888517
2040    628.426226
2050    689.505638
Name: baseline_total, dtype: float64

,baseline_total,avo_trees,avo_ac,avo_both,trees_pct,ac_pct,both_pct,trees_on_top_pct
year,,,,,,,,
2030,670.89,8.79,24.53,34.39,1.31,3.66,5.13,1.47
2040,628.43,8.47,20.23,32.39,1.35,3.22,5.15,1.93
2050,689.51,9.00,26.86,35.31,1.30,3.89,5.12,1.23


- Summary table: costs, avoided deaths, and EACs by policy

- We assemble a compact table with one row per policy case:
1) "Trees only (vs current AC)"
2) "AC only (vs current AC)"
3) "Both (trees + AC vs current AC)"
4) "Trees (incremental, on top of AC policy)"

For each case we report:
- PV_cost_eur:
  * For trees: PV_trees_total (CAPEX + O&M)
  * For AC:    PV_ac_total (CAPEX + maintenance + electricity)
  * For "Both": sum of trees and AC PVs
- avoided_deaths_cum:
  * Cumulative (undiscounted) avoided deaths over the 25-year horizon.
- Cost_per_avoided_death_eur:
  * PV_cost_eur divided by avoided_deaths_cum.
  * This is a present-value cost divided by an *undiscounted* number of avoided deaths.
    We keep this mixed metric for interpretability (it is closer to a "cost per life
    saved over the period").
- EAC_*_annuity_eur_per_yr:
  * Equivalent annual costs derived from the PVs and the annuity factor AF.
  * For trees we separate CAPEX and O&M; for AC we use a single total EAC.
- EAC_capex_paper_eur_per_yr:
  * Trees-only: "paper-style" annual cost based on the shortcut IR / ((1+r)^T * T).
    This is kept only for comparability with the working paper, the main EAC
    we rely on is the annuity-based one using the actual ramp.
- added_AC_users:
  * Total number of new AC users in the last year of the horizon under the AC policy (for
    rows that include AC). This table is the main quantitative output used in the text to
    compare:
  - Trees vs AC in terms of cost per avoided death and annual cost
  - The combined policy vs each single policy
  -  The incremental value of trees when an AC expansion is already in place.

In [15]:
def safe_ratio(c, b):
    return float(c / b) if (b is not None and b > 1e-9) else np.inf

summary_main = pd.DataFrame([
    {
        "Policy": "Trees only (vs current AC)",
        "PV_cost_eur": PV_trees_total,
        "avoided_deaths_cum": CUM_b_tree,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, CUM_b_tree),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": 0.0,
    },
    {
        "Policy": "AC only (vs current AC)",
        "PV_cost_eur": PV_ac_total,
        "avoided_deaths_cum": CUM_b_ac,
        "Cost_per_avoided_death_eur": safe_ratio(PV_ac_total, CUM_b_ac),
        "EAC_capex_annuity_eur_per_yr": 0.0,
        "EAC_om_annuity_eur_per_yr":    0.0,
        "EAC_total_annuity_eur_per_yr": EAC_ac_total,
        "EAC_capex_paper_eur_per_yr":   np.nan,
        "added_AC_users": added_users_final,
    },
    {
        "Policy": "Both (trees+AC vs current AC)",
        "PV_cost_eur": PV_trees_total + PV_ac_total,
        "avoided_deaths_cum": CUM_b_both,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total + PV_ac_total, CUM_b_both),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity + EAC_ac_total,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": added_users_final,
    },
    {
        # marginal trees effect if AC policy is already implemented
        "Policy": "Trees (incremental, on top of AC policy)",
        "PV_cost_eur": PV_trees_total,
        "avoided_deaths_cum": CUM_b_tree_cond,
        "Cost_per_avoided_death_eur": safe_ratio(PV_trees_total, CUM_b_tree_cond),
        "EAC_capex_annuity_eur_per_yr": EAC_capex_annuity,
        "EAC_om_annuity_eur_per_yr":    EAC_om_annuity,
        "EAC_total_annuity_eur_per_yr": EAC_total_annuity,
        "EAC_capex_paper_eur_per_yr":   EAC_capex_paper,
        "added_AC_users": 0.0,
    },
]).round(2)

summary_main

,Policy,PV_cost_eur,avoided_deaths_cum,Cost_per_avoided_death_eur,EAC_capex_annuity_eur_per_yr,EAC_om_annuity_eur_per_yr,EAC_total_annuity_eur_per_yr,EAC_capex_paper_eur_per_yr,added_AC_users
0,Trees only (vs current AC),5.213169e+08,170.73,3053498.31,12093350.58,17844769.69,29938120.28,5775851.59,0.00
1,AC only (vs current AC),6.973997e+08,592.32,1177400.89,0.00,0.00,40050178.02,NaN,189806.13
2,Both (trees+AC vs current AC),1.218717e+09,790.00,1542677.08,12093350.58,17844769.69,69988298.30,5775851.59,189806.13
3,"Trees (incremental, on top of AC policy)",5.213169e+08,197.68,2637179.02,12093350.58,17844769.69,29938120.28,5775851.59,0.00


**On a PV budget**

**Sensitivity**